In [1]:
import json
import os

meta_file = "/qumulo/shared_data/aofei_summer/data/BiomedParse_meta.json"

In [2]:
# If meta_file is provided, load multiple datasets
with open(meta_file, 'r') as f:
    meta = json.load(f)
roots = meta['roots']
mask_roots = meta['mask_roots']
json_paths = meta['json_paths']
modality_labels = meta['modality_labels']
assert len(roots) == len(mask_roots) == len(json_paths) == len(modality_labels)

imgid_to_anns = {}
image_infos = {}
categories = None
num_classes = None
all_annotations = []
for r, m, j, modality in zip(roots, mask_roots, json_paths, modality_labels):
    with open(j, 'r') as f:
        data = json.load(f)
    annotations = data['annotations']
    categories = data['categories']
    if categories is None:
        categories = categories
        num_classes = len(categories)
    # Build mapping from image_id to all masks and info
    for ann in annotations:
        img_id = ann['image_id']
        # Make img_id unique across datasets by prefixing with dataset index
        unique_img_id = f"{r}_{img_id}"
        ann['image_id'] = unique_img_id
        ann['file_name'] = os.path.join(r, ann['file_name'])
        ann['mask_file'] = os.path.join(m, ann['mask_file'])
        ann['modality_label'] = int(modality) if modality is not None else -1
        if unique_img_id not in imgid_to_anns:
            imgid_to_anns[unique_img_id] = []
        imgid_to_anns[unique_img_id].append(ann)
    all_annotations.extend(annotations)
    for ann in annotations:
        img_id = ann['image_id']
        if img_id not in image_infos:
            image_infos[img_id] = ann['file_name']

print(f"Total images: {len(imgid_to_anns)}, total annotations: {len(all_annotations)}")
# filter annotations to include more than 0 masks
filtered_annotations = [ann for ann in all_annotations if len(imgid_to_anns[ann['image_id']]) > 5 and len(imgid_to_anns[ann['image_id']]) <= 16]
print(f"Filtered annotations: {len(filtered_annotations)}")
annotations = filtered_annotations
image_ids = list(image_infos.keys())

Total images: 211411, total annotations: 448327
Filtered annotations: 74552


In [3]:
len(set([ann['image_id'] for ann in annotations]))  # number of modalities

8843

In [4]:
annotations[0]

{'mask_file': '/qumulo/shared_data/aofei_summer/data/BiomedParse/amos22/amos22/CT/train_mask/amos_0001_55_CT_abdomen_right+kidney.png',
 'area': 7661,
 'iscrowd': 0,
 'image_id': '/qumulo/shared_data/aofei_summer/data/BiomedParse/amos22/amos22/CT/train_17',
 'bbox': [323, 389, 112, 111],
 'category_id': 3,
 'id': 27,
 'slice_ratio': 0.7,
 'file_name': '/qumulo/shared_data/aofei_summer/data/BiomedParse/amos22/amos22/CT/train/amos_0001_55_CT_abdomen.png',
 'split': 'train',
 'sentences': [{'raw': 'right kidney in abdominal CT',
   'sent': 'right kidney in abdominal CT',
   'sent_id': 43}],
 'sent_ids': [43],
 'ann_id': 27,
 'ref_id': 27,
 'modality_label': 0}

In [5]:
s = 0
for i in annotations:
    if "amos" in i['file_name']:
        s += 1
s

74552

In [6]:
amos_filtered_image_names = list(set([ann['file_name'] for ann in annotations if "amos" in ann['file_name']]))
len(amos_filtered_image_names)

8843

In [7]:
# get the stats of numbers of different datasets
dataset_stats = {}
dataset_image_stats = {}
for ann in all_annotations:
    dataset_name = ann['file_name'].split('/')[6]
    image_id = ann['image_id']
    if dataset_name == "MSD":
        dataset_name = ann['file_name'].split('/')[-3]


    if dataset_name not in dataset_stats:
        dataset_stats[dataset_name] = 0
        dataset_image_stats[dataset_name] = []
    dataset_stats[dataset_name] += 1
    dataset_image_stats[dataset_name].append(image_id)

# print("Dataset statistics:")
# for ds, count in dataset_stats.items():
#     print(f"  {ds}: {count}") 

print("Dataset image statistics:")
for img, count in dataset_image_stats.items():
    print(f"  {img}: {len(set(count))}")

Dataset image statistics:
  amos22: 32274
  BreastUS: 519
  CAMUS: 17068
  CDD-CESM: 1016
  CXR_Masks_and_Labels: 544
  DRIVE: 15
  FH-PS-AOP: 3200
  G1020: 816
  GlaS: 123
  ISIC: 2694
  kits23: 24802
  LGG: 1014
  LIDC-IDRI: 7389
  LiverUS: 30
  MMs: 2880
  Task01_BrainTumour: 39791
  Task02_Heart: 959
  Task03_Liver: 12133
  Task04_Hippocampus: 4819
  Task05_Prostate: 728
  Task06_Lung: 1241
  Task07_Pancreas: 6662
  Task08_HepaticVessel: 7412
  Task09_Spleen: 784
  Task10_Colon: 1000
  NeoPolyp: 800
  OCT-CME: 1177
  PanNuke: 5143
  PolypGen: 1112
  COVID-19_CT: 1187
  COVID-QU-Ex: 4660
  QaTa-COV19: 7145
  Radiography: 16930
  REFUGE: 800
  siim-acr-pneumothorax: 2379
  UWaterlooSkinCancer: 165


In [22]:
# other candiates: BreastUS, 
# amos: 8843, panuke: 2000
165 + 2379 + 800 + 1112 + 2000 + 1177 + 800 + 1000 + 784 + 1500 + 1500 + 1241 + 728 + 1500 + 2000 + 959 + 3000 + 2000 + 30 + 2000 + 1014 + 2000 + 1000 + 123+ 816 + 1000 + 15 + 544 + 1016 + 2000 + 519 + 8000
# 260 + 2379 + 1600 + 1112 + 2000 + 1177 + 1640 + 1000 + 784 + 1000 + 1000 + 1241 + 1126 + 1000 + 2000 + 959 + 2000 + 2000 + 30 + 2000 + 2028 + 2000 + 1000 + 123 + 1448 + 1000 + 15 + 1632 + 1016 + 2000 + 8843

44722

In [8]:
dataset_image_stats.keys()
# sampled_nums = [8000, 519, 2000, 1016, 544, 15, 1000, 816, 123, 1000, 2000, 1014, 2000, 30, 1500, 3000, 959, 2000, 1500, 728, 1241, 2000, 1500, 784, 
#                 1000, 800, 1177, 2000, 1112, 200, 200, 200, 200, 800, 2000, 165]

sampled_nums = [16000, 519, 8000, 1016, 544, 15, 1600, 816, 123, 2000, 2000, 1014, 4000, 30, 2880, 8000, 959, 8000, 3000, 728, 1241, 4000, 4000, 784, 
                1000, 800, 1177, 4000, 1112, 200, 200, 200, 12000, 800, 2000, 165]
dataset_sampled_num = {k: v for k, v in zip(dataset_image_stats.keys(), sampled_nums)}

In [9]:
dataset_sampled_num

{'amos22': 16000,
 'BreastUS': 519,
 'CAMUS': 8000,
 'CDD-CESM': 1016,
 'CXR_Masks_and_Labels': 544,
 'DRIVE': 15,
 'FH-PS-AOP': 1600,
 'G1020': 816,
 'GlaS': 123,
 'ISIC': 2000,
 'kits23': 2000,
 'LGG': 1014,
 'LIDC-IDRI': 4000,
 'LiverUS': 30,
 'MMs': 2880,
 'Task01_BrainTumour': 8000,
 'Task02_Heart': 959,
 'Task03_Liver': 8000,
 'Task04_Hippocampus': 3000,
 'Task05_Prostate': 728,
 'Task06_Lung': 1241,
 'Task07_Pancreas': 4000,
 'Task08_HepaticVessel': 4000,
 'Task09_Spleen': 784,
 'Task10_Colon': 1000,
 'NeoPolyp': 800,
 'OCT-CME': 1177,
 'PanNuke': 4000,
 'PolypGen': 1112,
 'COVID-19_CT': 200,
 'COVID-QU-Ex': 200,
 'QaTa-COV19': 200,
 'Radiography': 12000,
 'REFUGE': 800,
 'siim-acr-pneumothorax': 2000,
 'UWaterlooSkinCancer': 165}

In [11]:
import random
all_Sampled_images = []
for ds, count in dataset_stats.items():
    all_image_names = list(set(dataset_image_stats[ds]))
    # if ds == "amos22":
    #     all_image_names_morethan5 = [img for img in all_image_names if len(imgid_to_anns[img]) >= 5]
    #     all_image_names_lessthan5 = [img for img in all_image_names if len(imgid_to_anns[img]) < 5]
    #     print(f"Dataset: {ds}, Images with >= 5 annotations: {len(all_image_names_morethan5)}, Images with < 5 annotations: {len(all_image_names_lessthan5)}")
    #     print(dataset_sampled_num[ds])
    #     sampled_images_morethan5 = random.sample(all_image_names_morethan5, 12000)
    #     sampled_images_lessthan5 = random.sample(all_image_names_lessthan5, 4000)
    #     sampled_images = sampled_images_morethan5 + sampled_images_lessthan5
    # else:
    if len(all_image_names) > dataset_sampled_num[ds]:
        sampled_images = random.sample(all_image_names, dataset_sampled_num[ds]//2)
    else:
        sampled_images = all_image_names
    all_Sampled_images.extend(sampled_images)

In [12]:
len(all_Sampled_images)

55323

In [16]:
# len(all_Sampled_images)
# all_Sampled_images[0]
step1_sampled_data = dict()
for k in all_Sampled_images:
    step1_sampled_data[k] = dict()
    step1_sampled_data[k]['image_file'] = image_infos[k]
    step1_sampled_data[k]['mask_annotations'] = imgid_to_anns[k]

In [17]:
# # Sample a subset of images and attach quantizer / modality info per mask
import random
from PIL import Image
import json
import torch

# random.seed(0)
# sample_size = 60000  # change as desired
# out_sample_file = "/qumulo/shared_data/aofei_summer/RegTok/data/RegAlign_data.json"

# image_ids = list(imgid_to_anns.keys())
# random.shuffle(image_ids)
# sampled_ids = image_ids[:sample_size]

In [ ]:
# step1_sampled_data = dict()
# for k in sampled_ids:
#     step1_sampled_data[k] = dict()
#     step1_sampled_data[k]['image_file'] = image_infos[k]
#     step1_sampled_data[k]['mask_annotations'] = imgid_to_anns[k]

In [18]:
len(step1_sampled_data), step1_sampled_data[k]

(26993,
 {'image_file': '/qumulo/shared_data/aofei_summer/data/BiomedParse/UWaterlooSkinCancer/UWaterlooSkinCancer/train/dermquest_SSM33_2_dermoscopy_skin.png',
  'mask_annotations': [{'mask_file': '/qumulo/shared_data/aofei_summer/data/BiomedParse/UWaterlooSkinCancer/UWaterlooSkinCancer/train_mask/dermquest_SSM33_2_dermoscopy_skin_melanoma.png',
    'area': 115986,
    'iscrowd': 0,
    'image_id': '/qumulo/shared_data/aofei_summer/data/BiomedParse/UWaterlooSkinCancer/UWaterlooSkinCancer/train_63',
    'bbox': [372, 388, 447, 377],
    'category_id': 12,
    'id': 126,
    'slice_ratio': 1.0,
    'file_name': '/qumulo/shared_data/aofei_summer/data/BiomedParse/UWaterlooSkinCancer/UWaterlooSkinCancer/train/dermquest_SSM33_2_dermoscopy_skin.png',
    'split': 'train',
    'sentences': [{'raw': 'melanoma', 'sent': 'melanoma', 'sent_id': 280}],
    'sent_ids': [280],
    'ann_id': 126,
    'ref_id': 126,
    'modality_label': 9},
   {'mask_file': '/qumulo/shared_data/aofei_summer/data/Biom

In [46]:
# load quantization models

import os
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

import sys

sys.path.append("/qumulo/shared_data/aofei_summer/RegTok/source")

os.environ['CUDA_VISIBLE_DEVICES'] = "2"

from tokenizer.vq_model import VQ_models

ckpt_path = "/qumulo/shared_data/aofei_summer/intern_records/RegTok/checkpoints/RegTok_quant_full_modal_rec_seg_modal/001-RegTok/checkpoints/0035673.pt"
region_ckpt_path = "/qumulo/shared_data/aofei_summer/intern_records/RegTok/checkpoints/RegTok_pipeline_full_wo_quant/002-RegTok/checkpoints/0079280.pt"

vq_model_name = "RegTok"
image_size = 192
batch_size = 2
device = "cuda" if torch.cuda.is_available() else "cpu"

/qumulo/shared_data/aofei_summer/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/qumulo/shared_data/aofei_summer/miniconda3/lib/python3.13/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [49]:
# Load model
vq_model = VQ_models[vq_model_name](
    codebook_size=32,
    num_stages=3,
    num_queries=20,
    codebook_embed_dim=64,
    dropout_p=0.0,
    kmeans=False,
    use_quantization=True,
    quant_use_seg=True,
    num_modalities=18,
    num_classes=17
)
ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
ckpt_region = torch.load(region_ckpt_path, map_location="cpu", weights_only=False)
# Load only matching keys (trainable params may be subset)
missing, unexpected = vq_model.load_state_dict(ckpt["model"], strict=False)
missing_region, unexpected_region = vq_model.load_state_dict(ckpt_region["model"], strict=False)
print("Missing keys:", [i for i in missing if ("image_encoder" not in i) and ("text_encoder" not in i)])
print("Unexpected keys:", unexpected)
print("Missing keys (region):", [i for i in missing_region if ("image_encoder" not in i) and ("text_encoder" not in i)])
print("Unexpected keys (region):", unexpected_region)
vq_model = vq_model.to(device)
vq_model.eval()

clip_preprocess = vq_model.unimed_preprocess

Codebook initialized with uniform distribution, allow gradient: False
Codebook initialized with uniform distribution, allow gradient: False
Codebook initialized with uniform distribution, allow gradient: False
Codebook initialized with uniform distribution, allow gradient: False
Codebook initialized with uniform distribution, allow gradient: False
Codebook initialized with uniform distribution, allow gradient: False
Codebook initialized with uniform distribution, allow gradient: False
Codebook initialized with uniform distribution, allow gradient: False
Codebook initialized with uniform distribution, allow gradient: False
Codebook initialized with uniform distribution, allow gradient: False
Codebook initialized with uniform distribution, allow gradient: False
Codebook initialized with uniform distribution, allow gradient: False
Codebook initialized with uniform distribution, allow gradient: False
Codebook initialized with uniform distribution, allow gradient: False
Codebook initialized

In [50]:
# Define transforms (should match training)
# Build dataset and split
import torchvision.transforms as T
image_size = 224
mask_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
])
def get_image_data(idx):
    img_id = idx
    img_file = image_infos[img_id]
    image = clip_preprocess(Image.open(img_file)).unsqueeze(0)

    # Get all masks for this image
    anns = imgid_to_anns[img_id]
    mask_list = []
    bbox_list = []
    category_list = []
    modality_list = []
    sentences_list = []
    for ann in anns:
        mask = Image.open(ann['mask_file']).convert('L')
        mask = T.ToTensor()(mask)
        mask = (mask > 0.5).float()
        mask_list.append(mask)
        bbox_list.append(torch.tensor(ann['bbox'], dtype=torch.float32))
        category_list.append(ann['category_id'])
        sentences_list.append([s['sent'] for s in ann.get('sentences', [])])
        modality_list.append(ann['modality_label']) # only use the first modality label
    modality_label = int(modality_list[0])

    if len(mask_list) > 0:
        masks = torch.stack(mask_list, dim=0)
    else:
        masks = torch.zeros((0, image_size, image_size), dtype=torch.float32)
    bboxes = torch.stack(bbox_list, dim=0) if bbox_list else torch.zeros((0, 4), dtype=torch.float32)
    categories = torch.tensor(category_list, dtype=torch.long) if category_list else torch.zeros((0,), dtype=torch.long)
    class_labels = categories
    # Encode sentences if text_encoder is provided
    text_embeddings = None
    
    return image, masks, class_labels, text_embeddings, modality_label


In [51]:
image, masks, class_labels, text_embeddings, modality_label = get_image_data(k)

In [52]:
def _detach_to_cpu(x):
    if torch.is_tensor(x):
        return x.detach().cpu()
    if isinstance(x, dict):
        return {k: _detach_to_cpu(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        t = [_detach_to_cpu(v) for v in x]
        return type(x)(t)
    return x

In [67]:
quantizer_infos = []
all_pred_classes = []
all_gt_classes = []
all_gt_masks = []
all_original_images = []

keys = all_Sampled_images
batch_size = 2
for k in tqdm(range(0, len(keys), batch_size)):
    batch_keys = keys[k : k + batch_size]
    images = []
    masks_batch = []
    class_labels_batch = []
    text_embeddings_batch = []
    modality_labels = []

    # load each sample in the batch
    for img_id in batch_keys:
        image, masks, class_labels, text_embeddings, modality_label = get_image_data(img_id)
        images.append(image)                       # (1, C, H, W)
        masks_batch.append(masks)                  # (num_masks, H, W) or (0,...)
        class_labels_batch.append(class_labels)    # (num_masks,) or (0,)
        text_embeddings_batch.append(text_embeddings)
        modality_labels.append(modality_label)

    imgs = torch.cat([item for item in images], dim=0)
    masks = [item.squeeze(1) for item in masks_batch]
    class_labels = [item for item in class_labels_batch]

    imgs = imgs.to(device)
    masks = [i.to(device) for i in masks]


    with torch.no_grad():
        outputs = vq_model(
            imgs, do_quantize=True, mask_labels=masks, class_labels=[c.to(device) for c in class_labels], loss_type="dice_bce"
        )
    # unpack outputs (adapt to your signature)
    *_, quantizer_info, semantic_loss = outputs[-2], outputs[-1]  # example; keep correct unpacking

    # detach/move quantizer_info to CPU before storing
    quantizer_infos.append(_detach_to_cpu(quantizer_info))

    # read needed fields from CPU-safe copy
    qinfo_cpu = quantizer_infos[-1]
    codebook_indices = qinfo_cpu['indices']        # already on CPU
    hungarian_indices = qinfo_cpu['hungarian_indices']
    modality_labels_pred = qinfo_cpu.get('modality_label_pred', None)

    # store only CPU numpy data for masks/images/labels
    for i in range(len(batch_keys)):
        all_original_images.append(imgs[i].cpu().numpy())   # move image to CPU
        all_gt_masks.append([m.cpu().numpy() for m in masks_batch[i]])  # masks_batch are CPU originally
        all_gt_classes.append(class_labels_batch[i].tolist() if isinstance(class_labels_batch[i], torch.Tensor) else [])

        # attach quantizer codes to annotations using CPU qinfo
        codebook_indices_per_image = codebook_indices[i] if isinstance(codebook_indices, (list, tuple)) else codebook_indices[i].numpy()
        hungarian_indices_per_image = hungarian_indices[i]
        modality_label_for_write_back = modality_labels_pred[i] if modality_labels_pred is not None else modality_labels[i]
        num_matched = hungarian_indices_per_image[0].shape[0]
        for p in range(num_matched):
            query_idx = int(hungarian_indices_per_image[0][p])
            codebook_idx = int(codebook_indices_per_image[query_idx])
            step1_sampled_data[batch_keys[i]]['mask_annotations'][p]['quantizer_code'] = f"M{modality_label_for_write_back}_{codebook_idx}"
    # free GPU refs explicitly
    del outputs, quantizer_info, imgs
    # torch.cuda.empty_cache()

  0%|          | 54/22572 [00:46<5:21:45,  1.17it/s]


KeyboardInterrupt: 

In [68]:
# save the results
out_sample_file = "/qumulo/shared_data/aofei_summer/RegTok/data/RegAlign_data_45k.json"
with open(out_sample_file, "w") as f:
    json.dump(step1_sampled_data, f, indent=2)

In [ ]:
# quantizer_infos = []
# all_pred_classes = []
# all_gt_classes = []
# all_gt_masks = []
# all_original_images = []

# keys = sampled_ids
# batch_size = 2
# for k in tqdm(range(0, len(keys), batch_size)):
#     batch_keys = keys[k : k + batch_size]
#     images = []
#     masks_batch = []
#     class_labels_batch = []
#     text_embeddings_batch = []
#     modality_labels = []

#     # load each sample in the batch
#     for img_id in batch_keys:
#         image, masks, class_labels, text_embeddings, modality_label = get_image_data(img_id)
#         images.append(image)                       # (1, C, H, W)
#         masks_batch.append(masks)                  # (num_masks, H, W) or (0,...)
#         class_labels_batch.append(class_labels)    # (num_masks,) or (0,)
#         text_embeddings_batch.append(text_embeddings)
#         modality_labels.append(modality_label)

#     imgs = torch.cat([item for item in images], dim=0)
#     masks = [item.squeeze(1) for item in masks_batch]
#     class_labels = [item for item in class_labels_batch]

#     imgs = imgs.to(device)
#     masks = [i.to(device) for i in masks]


#     with torch.no_grad():
#         outputs = vq_model(
#             imgs, do_quantize=True, mask_labels=masks, class_labels=[c.to(device) for c in class_labels], loss_type="dice_bce"
#         )
#         dec, diff, dice_loss, bce_loss, cls_loss, seg_logits, class_logits, hierarchical_codes, hierarchical_masks, hierarchical_gt_masks, hierarchical_losses, quantization_losses, \
#             total_quantization_loss, dice_loss_normal, dice_loss_quant, cls_loss_normal, cls_loss_quant, distill_loss, quantizer_info, semantic_loss = outputs
#         quantizer_infos.append(quantizer_info)
#         # print(quantizer_info)
#         # Optionally collect class predictions for analysis
#         if class_logits is not None:
#             all_pred_classes.append([logit.argmax().item() for logit in class_logits[0]])
#         #     all_gt_classes.append([c.item() for c in class_labels[0]])
#         # all_gt_masks.append([m.cpu().numpy() for m in masks[0]])
#         for b in range(len(class_labels)):
#             all_gt_classes.append([c.item() for c in class_labels[b]])
#             all_gt_masks.append([m.cpu().numpy() for m in masks[b]])
#             all_original_images.append(imgs[b].cpu().numpy())
#     codebook_indices = quantizer_info['indices']  # List of (batch_size, num_queries) per stage
#     hungarian_indices = quantizer_info['hungarian_indices']  # List of (batch_size, num_queries_matches) per stage
#     modality_labels_pred = quantizer_info['modality_label_pred']  # (batch_size,)
    
#     for m in range(batch_size):
#         image_id_for_write_back = batch_keys[m]
#         target_for_write_back = step1_sampled_data[image_id_for_write_back]

#         codebook_indices_per_image = codebook_indices[m].cpu().numpy()
#         hungarian_indices_per_image = hungarian_indices[m]
#         # num_matched = len(hungarian_indices_per_image) // 2
#         num_matched = hungarian_indices_per_image[0].shape[0]
#         modality_label_for_write_back = modality_labels_pred[m]
#         for p in range(num_matched):
#             query_idx = hungarian_indices_per_image[0].cpu().numpy()[p]
#             codebook_idx = codebook_indices_per_image[query_idx]
#             target_for_write_back['mask_annotations'][p]['quantizer_code'] = f"M{modality_label_for_write_back}_{codebook_idx}"


In [16]:
batch_keys, hungarian_indices,codebook_indices

(['/qumulo/shared_data/aofei_summer/data/BiomedParse/MSD/MSD/Task03_Liver/train_48',
  '/qumulo/shared_data/aofei_summer/data/BiomedParse/REFUGE/REFUGE/train_661'],
 [(tensor([ 3, 13]), tensor([0, 1])), (tensor([12, 13]), tensor([0, 1]))],
 tensor([[28, 16, 25, 16, 25, 25, 25, 28, 15, 16, 16, 16, 25, 25, 15, 25, 25, 25,
          25, 25],
         [19, 21, 15,  0,  6,  0, 11, 29, 20, 24, 21,  3,  0,  6, 20,  0, 21,  0,
           0,  6]]))

In [17]:
step1_sampled_data['/qumulo/shared_data/aofei_summer/data/BiomedParse/MSD/MSD/Task03_Liver/train_48']

{'image_file': '/qumulo/shared_data/aofei_summer/data/BiomedParse/MSD/MSD/Task03_Liver/train/liver_1_72_CT_liver.png',
 'mask_annotations': [{'mask_file': '/qumulo/shared_data/aofei_summer/data/BiomedParse/MSD/MSD/Task03_Liver/train_mask/liver_1_72_CT_liver_liver.png',
   'area': 53056,
   'iscrowd': 0,
   'image_id': '/qumulo/shared_data/aofei_summer/data/BiomedParse/MSD/MSD/Task03_Liver/train_48',
   'bbox': [422, 508, 399, 231],
   'category_id': 1,
   'id': 72,
   'slice_ratio': 0.3,
   'file_name': '/qumulo/shared_data/aofei_summer/data/BiomedParse/MSD/MSD/Task03_Liver/train/liver_1_72_CT_liver.png',
   'split': 'train',
   'sentences': [{'raw': 'liver in liver computed tomography',
     'sent': 'liver in liver computed tomography',
     'sent_id': 165},
    {'raw': 'liver in liver CT', 'sent': 'liver in liver CT', 'sent_id': 166},
    {'raw': 'hepatic organ in liver CT',
     'sent': 'hepatic organ in liver CT',
     'sent_id': 167}],
   'sent_ids': [165, 166, 167],
   'ann_id': 

In [40]:
target_for_write_back

{'image_file': '/qumulo/shared_data/aofei_summer/data/BiomedParse/amos22/amos22/CT/train/amos_0406_81_CT_liver.png',
 'mask_annotations': [{'mask_file': '/qumulo/shared_data/aofei_summer/data/BiomedParse/amos22/amos22/CT/train_mask/amos_0406_81_CT_liver_liver.png',
   'area': 14117,
   'iscrowd': 0,
   'image_id': '/qumulo/shared_data/aofei_summer/data/BiomedParse/amos22/amos22/CT/train_10389',
   'bbox': [323, 557, 207, 105],
   'category_id': 1,
   'id': 35816,
   'slice_ratio': 0.1,
   'file_name': '/qumulo/shared_data/aofei_summer/data/BiomedParse/amos22/amos22/CT/train/amos_0406_81_CT_liver.png',
   'split': 'train',
   'sentences': [{'raw': 'liver in liver CT',
     'sent': 'liver in liver CT',
     'sent_id': 70226}],
   'sent_ids': [70226],
   'ann_id': 35816,
   'ref_id': 35816,
   'modality_label': 0,
   'quantizer_code': 'M0_16'}]}